# TripSync AI — Notebook 04: Baseline Recommendation Models & Evaluation

**Development Day:** Day 6 of 35
**Author:** TripSync AI ML Engineering Team
**Objective:** Implement, benchmark, and mathematically verify baseline recommendation models:
1. **Random Heuristic Baseline** — Uniform stochastic sampling U(1.0, 5.0)
2. **Global Mean Baseline** — Unpersonalized central tendency y_bar_train
3. **Popularity Prior Baseline** — Non-personalized Bayesian attraction reputation prior
4. **Domain Heuristic Rule** — Zero-shot weighted interaction score
5. **Pure NumPy Mathematical Ridge Engine** — First-principles closed-form (X^T X + alpha*I)^(-1) X^T y solver
6. **Scikit-Learn Ridge Regressor** — Validated industrial reference model


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge

import sys
sys.path.append('..')
from src.math_engine import NumpyRidgeRegressor, evaluate_ranking_across_groups

# Configure visualization style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']


## 1. Load Processed Feature Splits and Feature Schema
We load the 17-feature interaction tensors generated in Day 5 partitioned strictly at the persona level.

In [ ]:
data_dir = Path('../data/processed')
train_df = pd.read_parquet(data_dir / 'train_features.parquet')
val_df = pd.read_parquet(data_dir / 'val_features.parquet')
test_df = pd.read_parquet(data_dir / 'test_features.parquet')

with open(data_dir / 'feature_schema.json', 'r') as f:
    schema = json.load(f)

feature_cols = schema['features']
target_col = schema['target_variable']

print(f'Train samples: {len(train_df)}, Val samples: {len(val_df)}, Test samples: {len(test_df)}')
print(f'Total features: {len(feature_cols)}')


## 2. Prepare Feature Matrices
Extract NumPy arrays for feature matrices X, target arrays y, and group vectors.

In [ ]:
X_train = train_df[feature_cols].values
y_train = train_df[target_col].values

X_val = val_df[feature_cols].values
y_val = val_df[target_col].values
groups_val = val_df['persona_id'].values

X_test = test_df[feature_cols].values
y_test = test_df[target_col].values
groups_test = test_df['persona_id'].values

print('Train X shape:', X_train.shape)
print('Test X shape:', X_test.shape)


## 3. Fit Pure NumPy Closed-Form Ridge Regressor vs. Scikit-Learn
The closed-form analytical equation is given by:
$$\mathbf{w}^* = (\mathbf{X}_{aug}^T \mathbf{X}_{aug} + \alpha \mathbf{I}')^{-1} \mathbf{X}_{aug}^T \mathbf{y}$$
where $\mathbf{X}_{aug} = [\mathbf{X}, \mathbf{1}]$ incorporates the intercept bias, and $\mathbf{I}'$ leaves the intercept unregularized.

In [ ]:
# Pure NumPy Implementation
numpy_ridge = NumpyRidgeRegressor(alpha=1.0)
numpy_ridge.fit(X_train, y_train)
y_pred_numpy = numpy_ridge.predict(X_test)

# Scikit-Learn Benchmark
sk_ridge = Ridge(alpha=1.0)
sk_ridge.fit(X_train, y_train)
y_pred_sklearn = sk_ridge.predict(X_test)

diff = np.max(np.abs(y_pred_numpy - y_pred_sklearn))
weight_diff = np.max(np.abs(numpy_ridge.weights - sk_ridge.coef_))
bias_diff = np.abs(numpy_ridge.bias - sk_ridge.intercept_)

print(f'Max prediction difference: {diff:.2e}')
print(f'Max weight difference:     {weight_diff:.2e}')
print(f'Bias intercept difference: {bias_diff:.2e}')


## 4. Run Baseline Benchmarking Pipeline
We load the official evaluation metrics produced by baseline_model.py and inspect test-set results.

In [ ]:
metrics_path = Path('../reports/baseline_metrics.json')
with open(metrics_path, 'r') as f:
    baseline_report = json.load(f)

summary_rows = []
for m_key, m_info in baseline_report['models'].items():
    t = m_info['test']
    summary_rows.append({
        'Model': m_info['model_name'],
        'Test RMSE': t['rmse'],
        'Test MAE': t['mae'],
        'Test R2': t['r2'],
        'NDCG@5': t['ndcg_at_5'],
        'NDCG@10': t['ndcg_at_10'],
        'Precision@5': t['precision_at_5'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


## 5. Visualizing Baseline Model Performance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Test RMSE Comparison
sns.barplot(data=summary_df, x='Test RMSE', y='Model', palette='mako', ax=ax1)
ax1.set_title('Test RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Root Mean Squared Error')

# 2. Ranking Quality (NDCG@5)
sns.barplot(data=summary_df, x='NDCG@5', y='Model', palette='viridis', ax=ax2)
ax2.set_title('Ranking Quality: NDCG@5 (Higher is Better)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Mean Normalized Discounted Cumulative Gain')
ax2.set_xlim(0, 1.05)

plt.tight_layout()
plt.show()


## 6. Learned Feature Weights Interpretation
Inspect which features received the largest linear weights in the Ridge recommendation engine.

In [ ]:
weights_data = baseline_report['top_learned_weights']
weights_df = pd.DataFrame(weights_data, columns=['Feature', 'Weight']).sort_values(by='Weight', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(weights_df['Feature'], weights_df['Weight'], color='royalblue', edgecolor='black')
plt.title('Learned Linear Weights (Ridge Regression Baseline)', fontsize=12, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()
